# Edge-AutoGuard: PHM Gearbox Dataset Inspection

This notebook documents the actual PHM Society gearbox dataset available in the local workspace before any modeling work.

Important note: the dataset currently present in the workspace is the raw PHM 2009 challenge run set. It contains sensor-time-series CSV files, but no explicit label column or metadata file was found in the downloaded files.

In [ ]:
import os
import glob
import pandas as pd

base_dir = r"c:/Users/kamal/Downloads/edge-autoguard/PHM09_competition_1"
files = sorted(glob.glob(os.path.join(base_dir, "*.csv")))
print(f"Total CSV files: {len(files)}")
print(f"First few files: {os.path.basename(files[0])}, {os.path.basename(files[1])}, {os.path.basename(files[2])}")

# Inspect the first file as a representative example.
first_file = files[0]
df = pd.read_csv(first_file, header=None)
print("\nShape of first file:", df.shape)
print("Sample rows from first file:")
print(df.head(5).to_string(index=False, header=False))

## 1) Dataset structure

The workspace contains 560 CSV files named like `Run_1.csv`, `Run_2.csv`, ..., `Run_560.csv`.
Each file has 3 columns and no header row.

The values in the first file are representative of the whole dataset:

- Column 0: a vibration/accelerometer-like signal
- Column 1: a second vibration/accelerometer-like signal
- Column 2: a tachometer-related signal (used to track rotational timing)

This is a raw time-series dataset, not a tabular dataset with one row per sample and one label column.

In [ ]:
# Summarize all CSV file shapes and show the file-level distribution.
shape_counts = {}
for f in files:
    df = pd.read_csv(f, header=None)
    shape_counts[df.shape] = shape_counts.get(df.shape, 0) + 1

print("Unique row/column shapes across the dataset:")
for shape, count in sorted(shape_counts.items()):
    print(f"  shape={shape}, files={count}")

# Show a few global statistics from the first 20 files combined.
combined = pd.concat([pd.read_csv(f, header=None) for f in files[:20]], ignore_index=True)
print("\nCombined statistics for first 20 files:")
print(combined.describe().round(6).to_string())

## 2) Observed file format and signal meaning

The PHM Society website describes the 2009 gearbox challenge as a synchronous vibration dataset collected from accelerometers mounted on the gearbox input and output shaft retaining plates. A tachometer provides zero-crossing information and rotational timing.

From the raw files, we observed:

- 560 CSV run files
- Each file is a time series of raw values
- 3 numeric columns
- No explicit header names
- No visible fault label column inside the CSV files
- No run-specific label metadata was found in the downloaded workspace folder

So the dataset should be treated as raw signal data that must be segmented and converted into ML samples using feature extraction.

## 3) Actual operating conditions

The public PHM challenge description says the runs were collected under different operating conditions, specifically:

- shaft speeds: approximately 30, 35, 40, 45, and 50 Hz
- loads: low and high

These operating conditions are part of the dataset collection design, but the downloaded CSV files themselves do not include a dedicated `speed`, `load`, or `condition` column.

## 4) Labels and fault categories

No explicit label file, class list, or per-run fault label was found in the downloaded workspace files.

Therefore, at this stage the actual dataset evidence is:

- raw gearbox sensor time series
- multiple runs across operating conditions
- no embedded labels in these files
- no explicit fault classes visible in the data folder

This means the project must either:

1. use an additional labeled dataset supplied separately, or
2. derive labels from external metadata not present here.

The current workspace does not contain that metadata, so the actual labels cannot be invented or assumed.

In [ ]:
# Print a representative sample of the first file in a compact, readable format.
first = pd.read_csv(files[0], header=None)
print("Representative first 10 values from Run_1.csv:")
print(first.head(10).to_string(index=False, header=False))

# Print the first row as a Python list for interpretation.
print("\nFirst row values:", first.iloc[0].tolist())

## 5) Interpretation of the dataset fields

Based on the PHM challenge description and actual CSV contents, the columns are interpreted as:

- First signal channel: gearbox vibration signal from the input-side accelerometer
- Second signal channel: gearbox vibration signal from the output-side accelerometer
- Third signal channel: tachometer signal, related to shaft rotation / timing information

These signals are not ready-made machine-learning rows. They are time-series recordings that would need to be segmented into windows and transformed into features such as RMS, standard deviation, kurtosis, skewness, peak-to-peak, and other time-domain statistics before training a classifier.

## 6) Conclusion for Phase 1

This inspection confirms that the dataset is a raw PHM 2009 gearbox signal dataset with the following actual properties:

- 560 CSV files
- 3 sensor channels per file
- no header row
- raw time-series numeric data
- no visible labels inside the downloaded files
- operating conditions described at the PHM challenge level (speed and load combinations), but not stored as explicit columns here

Because no actual labels or fault metadata were found in the data folder, the next phase must wait for confirmation on whether a separate labeled dataset is available or whether the task should proceed with the unlabeled raw PHM runs only.